# CRUX Step 6: Architecture C (Counterfactual Learning) and the controls A+CF and B+CF

This notebook runs the Step 6 plan on a Colab GPU: 3 architectures (C, A+CF, B+CF) x 1 trial x 3 seeds = 9 runs, with `budget.training_steps` steps each (`configs/protocol.yaml`). A and B come from the Step 5 rerun. Every run saves its files to Google Drive. After a disconnect, run the cells again: finished runs are skipped, and an unfinished run resumes from its last checkpoint. The notebook reads the train, dev_id, and validation splits only. It never reads the test split.

In [ ]:
# Mount Drive, clone the repo, and install the requirements.
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/0xAnto/crux.git /content/crux
%cd /content/crux
!pip install -q -r requirements.txt

The splits are not in git. This cell makes them again from the generator and the benchmark converters (as in `notebooks/step4_splits.ipynb`). The build stops with an error if a file has another SHA-256 than the frozen manifest. Then `load_split` checks every split against `configs/splits_manifest.json` and `configs/protocol.yaml` again.

In [ ]:
!python -m crux.data.boardgameqa
!python -m crux.data.dynabench
!python -m crux.data.supernat
!python -m crux.gen.generator
!python -m crux.splits build
!python -m crux.splits verify
from crux.splits import load_split
for name in ('train', 'dev_id', 'validation'):
    print(name, len(load_split(name)))  # raises ValueError if the hash differs

The sweep. `RUNS` is on Drive. A run is one (architecture, seed index). Put the Step 5 runs of A and B in `RUNS_STEP5` (12000 steps, trial 0). The aggregation compares all architectures with A.

In [ ]:
import os, subprocess, yaml
RUNS = '/content/drive/MyDrive/crux/runs/step6'
RUNS_STEP5 = '/content/drive/MyDrive/crux/runs/step5'
cfg = yaml.safe_load(open('configs/step6.yaml'))
protocol = yaml.safe_load(open('configs/protocol.yaml'))
print('steps per run:', protocol['budget']['training_steps'])
for arch in cfg['architectures']:  # C, A+CF, B+CF
    for seed in range(len(protocol['seeds'])):
        subprocess.run(['python', '-m', 'crux.train', '--config', 'configs/step6.yaml', '--arch', arch,
                        '--trial', '0', '--seed-index', str(seed), '--out', f'{RUNS}/{arch}/t0/s{seed}'], check=True)
for arch in ('A', 'B'):  # the Step 5 runs: the reference of the comparison
    if not os.path.exists(f'{RUNS}/{arch}'):
        os.symlink(f'{RUNS_STEP5}/{arch}', f'{RUNS}/{arch}')


The aggregation. It prints the mean and the standard deviation over the 3 seeds, the parameter counts, the latency, and the 95% paired bootstrap CI of the difference to A. The tables name the better direction in each column.

In [ ]:
!python -m crux.aggregate {RUNS}